# Viral Metagenomics Pipeline for Nanopore Sequencing Data
#=======================================

## Description:
This pipeline performs viral metagenomic analysis from Nanopore sequencing reads obtained from clinical samples.

## Workflow:
 1. Create Conda environments
 2. Raw read quality control
 3. Adapter trimming and read filtering
 4. Kraken2 database setup
 5. Taxonomic classification with Kraken2
 6. Interactive visualization with Krona

---

In [1]:
!pip install --upgrade --force-reinstall zstandard
!pip install -q condacolab

import condacolab
condacolab.install()

!sed -i '/cudatoolkit/d' /usr/local/conda-meta/pinned
!sed -i '/python/d' /usr/local/conda-meta/pinned

!mamba install -c conda-forge curl --quiet

  Using cached zstandard-0.25.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.3 kB)
Using cached zstandard-0.25.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (5.6 MB)
  Attempting uninstall: zstandard
    Found existing installation: zstandard 0.25.0
    Uninstalling zstandard-0.25.0:
      Successfully uninstalled zstandard-0.25.0
✨🍰✨ Everything looks OK!


# 1. Create conda environments

In [2]:
# Instala a ferramenta via PyPI
!pip install NanoFilt
!NanoFilt -v


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 125.7 MB/s eta 0:00:00
  Created wheel for NanoFilt: filename=NanoFilt-2.8.0-py3-none-any.whl size=8776 sha256=fe4ad7fcd7b0231e77116f9c1b4b1e3db2b35a0dfcea6772e4abb438f67fe31c
  Stored in directory: /root/.cache/pip/wheels/53/f3/c9/126ce1c746762c3454032c9e59ce8bc10290e78a1ab8dae8cc
Successfully built NanoFilt


NanoFilt 2.8.0


In [2]:
!mamba create -y -n nanopore_qc -c bioconda -c conda-forge nanoplot porechop nanofilt --quiet

!mamba create -y -n kraken_krona -c bioconda -c conda-forge kraken2 krona --quiet

!mamba install -y -c bioconda sra-tools --quiet

Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... 
Krona installed.  You still need to manually update the taxonomy
databases before Krona can generate taxonomic reports.  The update
script is ktUpdateTaxonomy.sh.  The default location for storing
taxonomic databases is /usr/local/envs/kraken_krona/opt/krona/taxonomy

If you would like the taxonomic data stored elsewhere, simply replace
this directory with a symlink.  For example:

rm -rf /usr/local/envs/kraken_krona/opt/krona/taxonomy
mkdir /path/on/big/disk/taxonomy
ln -s /path/on/big/disk/taxonomy /usr/local/envs/kraken_krona/opt/krona/taxonomy
ktUpdateTaxonomy.sh


done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done


# 2. Create directory structure and FASTQ file


In [3]:
# Directory
!mkdir -p analysis/{input,fastqc_pretrim,trimmed,fastqc_posttrim,kraken,krona}

# Download FASTQ file to input directory
!fasterq-dump ERR14817851
!gzip ERR14817851.fastq
!mv /content/ERR14817851.fastq.gz analysis/input/

spots read      : 117,928
reads read      : 117,928
reads written   : 117,928


# 3. Initial quality control with NanoPlot

In [4]:
# !mamba activate nanopore_qc
!mamba run -n nanopore_qc NanoPlot \
    --fastq analysis/input/ERR14817851.fastq.gz \
    -o analysis/fastqc_pretrim \
    -t 2

# Download
from google.colab import files
files.download("./analysis/fastqc_pretrim/NanoPlot-report.html")

# Open Nanoplot report
from IPython.display import HTML
with open("./analysis/fastqc_pretrim/NanoPlot-report.html", "r") as f:
    html = f.read()
HTML(html)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

General summary,
Mean read length,"1,056.2"
Mean read quality,15.6
Median read length,889.0
Median read quality,20.0
Number of reads,"117,928.0"
Read length N50,"1,185.0"
STDEV read length,624.4
Total bases,"124,550,071.0"
"Number, percentage and megabases of reads above quality cutoffs",
>Q10,108510 (92.0%) 116.0Mb


# 4. Adapter trimming and read filtering

In [5]:
# Running adapter trimming with Porechop
!mamba run -n nanopore_qc porechop \
    -i analysis/input/ERR14817851.fastq.gz \
    --threads 3 \
    -o analysis/trimmed/reads_trimmed.fastq.gz


Loading reads
analysis/input/ERR14817851.fastq.gz
117,928 reads loaded


Looking for known adapter sets

0 / 10,000 (0.0%)
10 / 10,000 (0.1%)
20 / 10,000 (0.2%)
30 / 10,000 (0.3%)
40 / 10,000 (0.4%)
50 / 10,000 (0.5%)
60 / 10,000 (0.6%)
70 / 10,000 (0.7%)
80 / 10,000 (0.8%)
90 / 10,000 (0.9%)
100 / 10,000 (1.0%)
110 / 10,000 (1.1%)
120 / 10,000 (1.2%)
130 / 10,000 (1.3%)
140 / 10,000 (1.4%)
150 / 10,000 (1.5%)
160 / 10,000 (1.6%)
170 / 10,000 (1.7%)
180 / 10,000 (1.8%)
190 / 10,000 (1.9%)
200 / 10,000 (2.0%)
210 / 10,000 (2.1%)
220 / 10,000 (2.2%)
230 / 10,000 (2.3%)
240 / 10,000 (2.4%)
250 / 10,000 (2.5%)
260 / 10,000 (2.6%)
270 / 10,000 (2.7%)
280 / 10,000 (2.8%)
290 / 10,000 (2.9%)
300 / 10,000 (3.0%)
310 / 10,000 (3.1%)
320 / 10,000 (3.2%)
330 / 10,000 (3.3%)
340 / 10,000 (3.4%)
350 / 10,000 (3.5%)
360 / 10,000 (3.6%)
370 / 10,000 (3.7%)
380 / 10,000 (3.8%)
390 / 10,000 (3.9%)
400 / 10,000 (4.0%)
410 / 10,000 (4.1%)
420 / 10,000 (4.2%)
430 / 10,000 (4.3%)
440 / 10,000 (4.4%)
450 /

In [6]:
!gunzip -c analysis/trimmed/reads_trimmed.fastq.gz | \
    NanoFilt -q 10 -l 900 --maxlength 1700 | \
    gzip > analysis/trimmed/ERR14817851_filtered.fastq.gz

In [7]:
!zcat analysis/trimmed/reads_trimmed.fastq.gz | wc -l
!zcat analysis/trimmed/ERR14817851_filtered.fastq.gz | wc -l

471668
163792


# 5. Final quality control with NanoPlot

In [8]:
!mamba run -n nanopore_qc NanoPlot \
    --fastq analysis/trimmed/ERR14817851_filtered.fastq.gz \
    -o analysis/fastqc_posttrim \
    -t 3

# Download
from google.colab import files
files.download("./analysis/fastqc_posttrim/NanoPlot-report.html")

# Open Nanoplot report
from IPython.display import HTML
with open("./analysis/fastqc_posttrim/NanoPlot-report.html", "r") as f:
    html = f.read()
HTML(html)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

General summary,
Mean read length,"1,205.4"
Mean read quality,18.1
Median read length,"1,164.0"
Median read quality,20.7
Number of reads,"40,948.0"
Read length N50,"1,220.0"
STDEV read length,219.0
Total bases,"49,360,227.0"
"Number, percentage and megabases of reads above quality cutoffs",
>Q10,40948 (100.0%) 49.4Mb


# 6. Kraken2 database setup

In [9]:
# You still need to manually update the taxonomy databases before Krona can
# generate taxonomic reports.  The update script is ktUpdateTaxonomy.sh.
# The default location for storing taxonomic databases is
#/usr/local/envs/kraken_krona/opt/krona/taxonomy

!mamba run -n kraken_krona ktUpdateTaxonomy.sh

# Download Kraken2 taxonomy
!mamba run -n kraken_krona kraken2-build --download-library viral --db viral_DB --use-ftp

Fetching taxdump.tar.gz...
   Fetching checksum...
   Checksum for taxdump.tar.gz matches server.
Extracting taxonomy...

Cleaning up...

Finished.




In [10]:
# Taxonomic classification with Kraken2
!mamba run -n kraken_krona kraken2 \
    --db viral_DB \
    --report analysis/kraken/kraken_report.txt \
    analysis/trimmed/ERR14817851_filtered.fastq.gz \
    --gzip \
    --output analysis/kraken/kraken_output.txt \
    --threads 3 \
    --confidence 0.1

# Download
from google.colab import files
files.download("./analysis/kraken/kraken_output.txt")

from google.colab import files
files.download("./analysis/kraken/kraken_report.txt")

Loading database information... done.
40948 sequences (49.36 Mbp) processed in 6.555s (374.8 Kseq/m, 451.83 Mbp/m).
  38993 sequences classified (95.23%)
  1955 sequences unclassified (4.77%)



# 8. Interactive visualization with Krona

In [11]:
!cut -f2,3 analysis/kraken/kraken_output.txt > \
    analysis/krona/kraken_krona_input.txt

!mamba run -n kraken_krona ktImportTaxonomy \
    analysis/krona/kraken_krona_input.txt \
    -o analysis/krona/krona_report.html

# Download
from google.colab import files
files.download("./analysis/krona/krona_report.html")

Loading taxonomy...
Importing analysis/krona/kraken_krona_input.txt...
Writing analysis/krona/krona_report.html...

   [ WARNING ]  Too many query IDs to store in chart; storing supplemental
                files in 'analysis/krona/krona_report.html.files'.

